# 02 - Confronto sistematico sui carichi prandiali (paziente fisso: adult#001)

**Obiettivo.** Confrontare il PID di riferimento e il controllore proposto su pasti non annunciati da 30, 60 e 90 g,
mantenendo fisso il paziente. Si confronta anche il **PID cons.**, che ottiene la sicurezza riducendo l'azione integrale
invece che con vincoli espliciti, e il **PID senza AW** della prima versione come riferimento del windup.

**Condizioni identiche.** Stesso paziente, sensore e seme del rumore, stesse condizioni iniziali e assestamento,
stessa durata e passo, stesso pasto, stessi guadagni e limiti dell'attuatore. Tra `PID` e `PID+IOB+PLGS` cambia
solo l'attivazione del tetto IOB e del PLGS (verificato nella tabella 1.2).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import aid_sim as A, notebook_utils as U
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110)
SEED = 1                                  # seme del rumore del sensore (uguale per tutti i controllori)
COL = {"PID senza AW": "tab:gray", "PID": "tab:red", "PID cons.": "tab:orange",
       "PID+PLGS": "tab:green", "PID+IOB": "tab:purple", "PID+IOB+PLGS": "tab:blue"}
print("Versioni delle librerie usate:"); display(U.library_versions())

In [ ]:
def finestra(df):
    """Solo la finestra di analisi (08:00-24:00): l'assestamento 00:00-08:00 non entra né nei grafici né nelle metriche."""
    return df[df.t_h >= A.WARMUP_H]

def decora(ax, meal_h=A.MEAL_CLOCK_H, banda=True):
    """Linea del pasto e griglia; banda 70-180 mg/dL solo sui pannelli di glicemia."""
    if banda:
        ax.axhspan(70, 180, color="green", alpha=0.12)
    ax.axvline(meal_h, color="k", ls=":", lw=1.2); ax.grid(ls=":", alpha=0.5)

def tabella(risultati, chiavi):
    """risultati: dict {chiave: DataFrame}; chiavi: nomi delle colonne-indice per la tabella."""
    rows = []
    for k, df in risultati.items():
        k = k if isinstance(k, tuple) else (k,)
        rows.append({**dict(zip(chiavi, k)), **{m: round(float(v), 1) for m, v in A.metrics(df).items()}})
    return pd.DataFrame(rows)

## 1. Configurazione
### 1.1 Scenario

In [ ]:
PAZIENTE = "adult#001"
ub = U.initial_conditions(PAZIENTE, SEED)["ub"]
display(U.scenario_table(PAZIENTE, "30, 60 o 90", SEED))

### 1.2 Parametri dei controllori affiancati (differenze evidenziate)

In [ ]:
NOMI = ["PID senza AW", "PID", "PID cons.", "PID+IOB+PLGS"]
tabs = {n: U.controller_table(A.CFG[n], ub).set_index("Parametro") for n in NOMI}
cmp_ = pd.DataFrame({n: tabs[n]["Valore"].astype(str) for n in NOMI}); cmp_["Unita'"] = tabs["PID"]["Unita'"]
display(cmp_)
diff = [p for p in cmp_.index if cmp_.loc[p, "PID"] != cmp_.loc[p, "PID+IOB+PLGS"]]
print("Parametri diversi tra PID e PID+IOB+PLGS:", diff)

### Unità di misura delle variabili salvate
| Variabile | Unità | Significato |
|---|---|---|
| `t_h` | h | ora del giorno (la simulazione parte alle 00:00) |
| `CGM` | mg/dL | misura del sensore, con rumore: è l'unica grandezza vista dal controllore |
| `BG` | mg/dL | glicemia plasmatica del modello UVa/Padova (Gp/Vg), usata per le metriche |
| `Gsub` | mg/dL | glicemia sottocutanea senza rumore (`info['bg']` di simglucose) |
| `Insulin_Uh` | U/h | velocità di infusione comandata |
| `IOB` | U | insulina attiva netta stimata dal controllore (eccesso rispetto al basale) |
| `IOB_vero` | U | insulina sottocutanea netta del modello (Isc1 + Isc2 meno l'equilibrio basale) |
| `Vincolo` | - | vincolo attivo nel passo: `iob`, `plgs`, `sat_alta`, `sat_bassa` o vuoto |

Metriche: TIR, TAR, TBR in % del tempo nella finestra 08:00-24:00; `TIR_post6h` nelle 6 h dopo il pasto;
`TBR_tardivo` dalle 3 h dopo il pasto alle 24:00; picco e nadir in mg/dL; `insulina_tot` in U nella finestra.

## 2. Simulazioni

In [ ]:
CARICHI = [30.0, 60.0, 90.0]
R = {(int(c), n): A.simulate(PAZIENTE, c, A.CFG[n], seed=SEED)[0] for c in CARICHI for n in NOMI}
print(len(R), "simulazioni completate")

## 3. Grafici: glicemia, infusione e IOB per ciascun carico

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(15, 9), sharex=True)
for j, c in enumerate(CARICHI):
    for n in NOMI:
        d = finestra(R[(int(c), n)])
        ax[0, j].plot(d.t_h, d.BG, color=COL[n], label=n)
        ax[1, j].plot(d.t_h, d.Insulin_Uh, color=COL[n], drawstyle="steps-post")
        ax[2, j].plot(d.t_h, d.IOB, color=COL[n])
    ax[2, j].axhline(1.5 * ub * 60, color="gray", ls="-.")
    ax[0, j].set_title(f"{int(c)} g di carboidrati")
    decora(ax[0, j]); decora(ax[1, j], banda=False); decora(ax[2, j], banda=False)
    ax[2, j].set_xlabel("Ora del giorno [h]")
ax[0, 0].set_ylabel("BG plasmatica [mg/dL]"); ax[1, 0].set_ylabel("Infusione [U/h]"); ax[2, 0].set_ylabel("IOB stimato [U]")
ax[0, 0].legend(fontsize=8); ax[2, 0].set_xlim(A.WARMUP_H, A.ANALYSIS_END_H); plt.tight_layout(); plt.show()

## 4. Indicatori clinici

In [ ]:
T = tabella(R, ["CHO [g]", "Controllore"]).sort_values(["CHO [g]", "Controllore"]).reset_index(drop=True)
display(T)
P = T[T.Controllore == "PID"].set_index("CHO [g]").drop(columns="Controllore")
Q = T[T.Controllore == "PID+IOB+PLGS"].set_index("CHO [g]").drop(columns="Controllore")
print("Differenza (PID+IOB+PLGS - PID):"); display((Q - P).round(1))

## 5. Contributo dei singoli moduli (60 g)
Tetto IOB e PLGS attivati separatamente e insieme, a parità di PID.

In [ ]:
ABL = ["PID", "PID+PLGS", "PID+IOB", "PID+IOB+PLGS"]
Rab = {n: A.simulate(PAZIENTE, 60.0, A.CFG[n], seed=SEED)[0] for n in ABL}
display(tabella(Rab, ["Controllore"]))

## 6. Lettura dei risultati (calcolata dalla tabella)

In [ ]:
for c in [30, 60, 90]:
    p = T[(T["CHO [g]"] == c) & (T.Controllore == "PID")].iloc[0]; q = T[(T["CHO [g]"] == c) & (T.Controllore == "PID+IOB+PLGS")].iloc[0]
    k = T[(T["CHO [g]"] == c) & (T.Controllore == "PID cons.")].iloc[0]
    print(f"{c:2d} g | TIR  PID {p.TIR:5.1f}  cons. {k.TIR:5.1f}  proposto {q.TIR:5.1f} % | "
          f"TBR  PID {p.TBR:5.1f}  cons. {k.TBR:5.1f}  proposto {q.TBR:5.1f} % | nadir {p.nadir:4.0f} / {k.nadir:4.0f} / {q.nadir:4.0f} mg/dL")

**Nota sul seme.** Questo notebook usa un solo seme del rumore del sensore. La robustezza rispetto al rumore
(5 semi, 60 g, adult#001 e adult#003) è in `results/robustezza_semi.csv`, generato da `run_campaign.py`.